In [1]:
"""SESSION 19 - YOLO11x-seg: 5-fold CV + multi-seed + fold ensembling.
Kaggle T4x2 'Save and Run All' — attach Image_Annotation_rebuilt.zip, enable Internet.
"""
from __future__ import annotations

import argparse
import json
import math
import random
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
try:
    import yaml
except ImportError:
    import subprocess as _sp, sys as _sys
    _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    import yaml

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
RUNTIME_BUILD = "SESSION19-YOLO11X-SEG-RUNTIME-V3"


# --------------------------------------------------------------------------- #
# GPU / device auto-detection
# --------------------------------------------------------------------------- #
def _auto_detect_device() -> str:
    """Return the best device string for Kaggle 'Save and Run All'.

    Returns '0,1' if two CUDA GPUs are visible, '0' for one GPU, 'cpu'
    as final fallback.  Never raises.
    """
    try:
        import torch
        n = torch.cuda.device_count()
        if n >= 2:
            print(f"  [device] {n} GPUs detected → using '0,1' (DDP)")
            return "0,1"
        if n == 1:
            print("  [device] 1 GPU detected → using '0'")
            return "0"
    except Exception:
        pass
    print("  [device] No GPU detected → using 'cpu'")
    return "cpu"


# --------------------------------------------------------------------------- #
# auto-detect / auto-build the rebuilt dataset
# --------------------------------------------------------------------------- #
def _resolve_data_root(requested: str, auto_rebuild: bool, rebuild_out: str) -> Path:
   
    # 1. Explicit / default path
    p = Path(requested).resolve()
    if (p / "data.yaml").is_file():
        return p

    # 2. Uploaded rebuilt dataset anywhere under /kaggle/input
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for candidate in sorted(kaggle_input.rglob("skin_disease_yolo")):
            if candidate.is_dir() and (candidate / "data.yaml").is_file():
                print(f"  [auto-detect] Found rebuilt dataset at: {candidate}")
                return candidate.resolve()

        # 3. Any data.yaml whose sibling directories look like a rebuilt layout
        for dy in sorted(kaggle_input.rglob("data.yaml")):
            root = dy.parent
            if all((root / s / "images").is_dir() for s in ("train", "valid", "test")):
                print(f"  [auto-detect] Found rebuilt layout at: {root}")
                return root.resolve()

        # 4. Auto-rebuild from raw Roboflow export
        if auto_rebuild:
            raw_yamls = sorted(kaggle_input.rglob("data.yaml"))
            if raw_yamls:
                raw_yaml = raw_yamls[0]
                print(f"  [auto-rebuild] Rebuilt dataset not found.")
                print(f"  [auto-rebuild] Running SESSION_00_rebuild_dataset.py on: {raw_yaml}")
                # Locate SESSION_00 script: same dir as this script, or kaggle/input.
                s00 = Path(__file__).parent / "SESSION_00_rebuild_dataset.py"
                if not s00.is_file():
                    for s00c in sorted(kaggle_input.rglob("SESSION_00_rebuild_dataset.py")):
                        s00 = s00c
                        break
                if not s00.is_file():
                    raise SystemExit(
                        "[auto-rebuild] Could not find SESSION_00_rebuild_dataset.py. "
                        "Add it to your Kaggle dataset or pass --data-root explicitly."
                    )
                import subprocess as _sp
                result = _sp.run(
                    [sys.executable, str(s00),
                     "--data-yaml", str(raw_yaml),
                     "--out-dir",   rebuild_out],
                    check=False
                )
                if result.returncode != 0:
                    raise SystemExit(
                        f"[auto-rebuild] SESSION_00_rebuild_dataset.py failed "
                        f"(exit {result.returncode}). Check the output above."
                    )
                built = Path(rebuild_out).resolve()
                if (built / "data.yaml").is_file():
                    return built

    # Nothing worked — give an actionable error.
    raise SystemExit(
        f"\n"
        f"================================================================\n"
        f"SESSION 19 ERROR: Rebuilt dataset not found.\n"
        f"================================================================\n"
        f"Looked for : {p}\n"
        f"\n"
        f"Fix options (pick one):\n"
        f"\n"
        f"  Option A — Run SESSION 00 first in this notebook session:\n"
        f"    !python SESSION_00_rebuild_dataset.py --out-dir /kaggle/working/Image_Annotation_rebuilt\n"
        f"  Then re-run SESSION 19.\n"
        f"\n"
        f"  Option B — Let SESSION 19 auto-run SESSION 00:\n"
        f"    !python SESSION_19_yolo11l_seg_cv_ensemble.py --auto-rebuild\n"
        f"\n"
        f"  Option C — Upload the rebuilt folder as a Kaggle dataset and attach it.\n"
        f"    SESSION 19 will auto-detect it under /kaggle/input/.\n"
        f"\n"
        f"  Option D — Pass the correct path explicitly:\n"
        f"    !python SESSION_19_yolo11l_seg_cv_ensemble.py --data-root /path/to/Image_Annotation_rebuilt\n"
        f"================================================================"
    )


# --------------------------------------------------------------------------- #
# dataset loading
# --------------------------------------------------------------------------- #
def load_names(data_yaml: Path) -> list[str]:
    data = yaml.safe_load(data_yaml.read_text())
    names = data.get("names")
    if isinstance(names, dict):
        return [str(names[k]) for k in sorted(names, key=lambda k: int(k))]
    if isinstance(names, list):
        return [str(n) for n in names]
    raise ValueError("data.yaml needs a 'names' list or mapping.")


def gather_split(root: Path, split: str) -> list[dict[str, Any]]:
    """Return [{img, stem, label}] for a rebuilt split (images with >=1 label)."""
    img_dir = root / split / "images"
    lbl_dir = root / split / "labels"
    if not img_dir.is_dir():
        return []
    items = []
    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        lbl = lbl_dir / f"{img.stem}.txt"
        if lbl.is_file() and lbl.read_text().strip():
            items.append({"img": img, "stem": img.stem, "label": lbl})
    return items


def multi_hot(items: list[dict[str, Any]], n_classes: int) -> np.ndarray:
    Y = np.zeros((len(items), n_classes), dtype=int)
    for i, it in enumerate(items):
        for line in it["label"].read_text().splitlines():
            if line.strip():
                c = int(line.split()[0])
                if 0 <= c < n_classes:
                    Y[i, c] = 1
    return Y


# --------------------------------------------------------------------------- #
# multilabel-stratified k-fold (+ fallback + presence repair)
# --------------------------------------------------------------------------- #
def stratified_kfold(Y: np.ndarray, k: int, seed: int) -> np.ndarray:
    n = len(Y)
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
        mskf = MultilabelStratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        fold_of = np.empty(n, dtype=int)
        for f, (_, val_idx) in enumerate(mskf.split(np.zeros((n, 1)), Y)):
            fold_of[val_idx] = f
        return fold_of
    except Exception as err:
        print(f"  (iterative-stratification unavailable: {err}; greedy fallback)")
        rng = random.Random(seed)
        order = list(range(n))
        freq = Y.sum(axis=0)
        rarity = Y @ (1.0 / np.maximum(freq, 1))
        order.sort(key=lambda i: (-rarity[i], rng.random()))
        counts = np.zeros(k)
        fold_of = np.zeros(n, dtype=int)
        for i in order:
            f = int(np.argmin(counts))
            fold_of[i] = f
            counts[f] += 1
        return fold_of


def repair_presence(fold_of: np.ndarray, Y: np.ndarray, k: int) -> np.ndarray:
    """Guarantee every class appears in every fold.

    Memory-efficient: caches membership arrays and updates only the two
    touched folds per move, avoiding O(1000*k) repeated np.where() scans
    that can OOM-kill the Kaggle kernel.
    """
    K = Y.shape[1]
    changed, guard = True, 0
    # Build membership cache once; update incrementally on each move.
    members: list[np.ndarray] = [np.where(fold_of == f)[0] for f in range(k)]
    while changed and guard < 200:
        changed, guard = False, guard + 1
        for f in range(k):
            if len(members[f]) == 0:
                continue
            present = Y[members[f]].sum(axis=0)
            for c in range(K):
                if present[c] > 0:
                    continue
                best_count, best_img_idx, best_d = 0, -1, -1
                for d in range(k):
                    if d == f:
                        continue
                    c_mask = Y[members[d], c] > 0
                    n_have = int(c_mask.sum())
                    if n_have > 1 and n_have > best_count:
                        best_count = n_have
                        best_img_idx = int(members[d][c_mask][0])
                        best_d = d
                if best_d == -1:
                    continue
                fold_of[best_img_idx] = f
                members[best_d] = members[best_d][members[best_d] != best_img_idx]
                members[f] = np.append(members[f], best_img_idx)
                present[c] = 1
                changed = True
    return fold_of


def write_fold_files(work: Path, items: list[dict[str, Any]], fold_of: np.ndarray,
                     k: int, names: list[str]) -> list[Path]:
    yamls = []
    folds_dir = work / "folds"
    for f in range(k):
        fdir = folds_dir / f"fold_{f}"
        fdir.mkdir(parents=True, exist_ok=True)
        val_idx = np.where(fold_of == f)[0]
        tr_idx = np.where(fold_of != f)[0]
        (fdir / "train.txt").write_text(
            "\n".join(str(items[i]["img"].resolve()) for i in tr_idx) + "\n")
        (fdir / "valid.txt").write_text(
            "\n".join(str(items[i]["img"].resolve()) for i in val_idx) + "\n")
        y = {"train": str((fdir / "train.txt").resolve()),
             "val": str((fdir / "valid.txt").resolve()),
             "nc": len(names), "names": {i: n for i, n in enumerate(names)}}
        (fdir / "data.yaml").write_text(yaml.safe_dump(y, sort_keys=False))
        yamls.append(fdir / "data.yaml")
    return yamls


# --------------------------------------------------------------------------- #
# training / validation (Ultralytics)
# --------------------------------------------------------------------------- #
def train_fold(model_arch: str, data_yaml: Path, run_name: str, args) -> dict[str, Any]:
    """Train one model; return {best_pt, val_map50, val_map, per_class}."""
    from ultralytics import YOLO

    run_dir = Path(args.work_dir) / "runs" / run_name
    best_pt = run_dir / "weights" / "best.pt"
    if args.skip_existing and best_pt.is_file():
        print(f"  [skip] {run_name}: best.pt exists, re-validating only")
    else:
        model = YOLO(model_arch)
        model.train(
            data=str(data_yaml), epochs=args.epochs, imgsz=args.imgsz,
            batch=args.batch, device=args.device, seed=args.seed_for_run,
            project=str(Path(args.work_dir) / "runs"), name=run_name,
            exist_ok=True, patience=args.patience, cache=args.cache,
            close_mosaic=args.close_mosaic, copy_paste=args.copy_paste,
            mosaic=args.mosaic, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            fliplr=0.5, flipud=0.0, degrees=args.degrees, translate=0.1,
            scale=0.5, amp=True, verbose=False, plots=False,
            time=args.fold_hours,  # hard per-fold wall-clock cap (hours)
        )
    model = YOLO(str(best_pt))
    metrics = model.val(data=str(data_yaml), split="val", imgsz=args.imgsz,
                        device=_single_device(args.device), verbose=False)
    rd = getattr(metrics, "results_dict", {}) or {}
    map50 = float(rd.get("metrics/mAP50(M)", getattr(metrics.seg, "map50", float("nan"))))
    map5095 = float(rd.get("metrics/mAP50-95(M)", getattr(metrics.seg, "map", float("nan"))))
    per_class = {}
    try:
        for ci, c in enumerate(metrics.seg.ap_class_index):
            per_class[str(int(c))] = float(metrics.seg.ap50[ci])
    except Exception:
        pass
    return {"best_pt": str(best_pt), "val_map50": map50,
            "val_map50_95": map5095, "per_class_map50": per_class}


def _single_device(device: str) -> str:
    """Ensemble prediction / val runs on one GPU; take the first of a DDP list."""
    return str(device).split(",")[0].strip() if device not in ("cpu", "") else "cpu"


# --------------------------------------------------------------------------- #
# prediction + mask fusion (weighted-box-fusion style, with soft masks)
# --------------------------------------------------------------------------- #
def predict_instances(best_pt: str, items: list[dict[str, Any]], args) -> dict[str, list]:
    """Run one model over items. Returns stem -> list of dets.

    Each det: {box:[x1,y1,x2,y2] px, score, cls, poly:[(x,y),...] px, wh:(W,H)}.
    """
    from ultralytics import YOLO
    model = YOLO(best_pt)
    out: dict[str, list] = {}
    # Ultralytics 8.3.x converts a Python list of paths into one in-memory
    # image batch. For a full holdout that can exhaust a T4. A .txt source
    # uses the normal batched image loader and preserves the image path.
    list_dir = Path(args.work_dir) / "prediction_lists"
    list_dir.mkdir(parents=True, exist_ok=True)
    source_list = list_dir / "test_images.txt"
    source_list.write_text("\n".join(str(it["img"].resolve()) for it in items) + "\n")
    stem_by_path = {str(it["img"].resolve()): it["stem"] for it in items}
    results = model.predict(source=str(source_list), imgsz=args.imgsz, conf=args.conf,
                            iou=args.pred_iou, device=_single_device(args.device),
                            batch=args.predict_batch, retina_masks=True,
                            stream=True, verbose=False)
    for r in results:
        stem = stem_by_path.get(str(Path(r.path).resolve()))
        if stem is None:
            raise RuntimeError(f"Prediction returned an unknown image path: {r.path}")
        dets = []
        W, H = r.orig_shape[1], r.orig_shape[0]
        if r.masks is not None and r.boxes is not None and len(r.boxes) > 0:
            boxes = r.boxes.xyxy.cpu().numpy()
            scores = r.boxes.conf.cpu().numpy()
            clss = r.boxes.cls.cpu().numpy().astype(int)
            polys = r.masks.xy  # list of (n,2) arrays in original-image pixels
            for b, s, c, p in zip(boxes, scores, clss, polys):
                if p is None or len(p) < 3:
                    continue
                dets.append({"box": b.tolist(), "score": float(s), "cls": int(c),
                             "poly": [(float(x), float(y)) for x, y in p],
                             "wh": (int(W), int(H))})
        out[stem] = dets
    expected_stems = {it["stem"] for it in items}
    if set(out) != expected_stems:
        raise RuntimeError("Prediction did not produce exactly one result for every requested image.")
    return out


def _iou_xyxy(a: list[float], b: list[float]) -> float:
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def _raster(poly: list[tuple[float, float]], W: int, H: int) -> np.ndarray:
    """Rasterise a polygon to a HxW uint8 mask using PIL (no cv2 dependency)."""
    from PIL import Image, ImageDraw
    im = Image.new("L", (W, H), 0)
    ImageDraw.Draw(im).polygon(poly, outline=1, fill=1)
    return np.asarray(im, dtype=np.uint8)


def fuse_image(dets_per_model: list[list[dict]], n_models: int, args) -> list[dict]:
    """Weighted-box-fusion-style merge across models for a single image.

    Cluster same-class detections by box IoU, then average box + soft-average
    masks. Cluster score = sum(member scores)/n_models (rewards agreement).
    """
    alld = []
    for dets in dets_per_model:
        alld.extend(dets)
    if not alld:
        return []
    alld.sort(key=lambda d: d["score"], reverse=True)
    used = [False] * len(alld)
    fused = []
    for i in range(len(alld)):
        if used[i]:
            continue
        cluster = [alld[i]]
        used[i] = True
        for j in range(i + 1, len(alld)):
            if used[j] or alld[j]["cls"] != alld[i]["cls"]:
                continue
            if _iou_xyxy(alld[i]["box"], alld[j]["box"]) >= args.wbf_iou:
                cluster.append(alld[j])
                used[j] = True
        support = len(cluster)
        if support < args.min_support:
            continue
        weights = np.array([d["score"] for d in cluster])
        boxes = np.array([d["box"] for d in cluster])
        fbox = (boxes * weights[:, None]).sum(0) / weights.sum()
        fscore = float(weights.sum() / n_models)
        W, H = cluster[0]["wh"]
        acc = np.zeros((H, W), dtype=np.float32)
        for d, w in zip(cluster, weights):
            acc += w * _raster(d["poly"], W, H)
        soft = acc / weights.sum()
        mask = (soft >= 0.5).astype(np.uint8)
        if mask.sum() == 0:
            continue
        fused.append({"box": fbox.tolist(), "score": min(1.0, fscore),
                      "cls": cluster[0]["cls"], "mask": mask, "wh": (W, H)})
    return fused


# --------------------------------------------------------------------------- #
# COCO ground truth / detections + evaluation (pycocotools segm)
# --------------------------------------------------------------------------- #
def build_coco_gt(items: list[dict[str, Any]], names: list[str],
                  dest: Path) -> tuple[dict, dict[str, int]]:
    from PIL import Image as PImage
    images, annotations = [], []
    stem_to_id: dict[str, int] = {}
    ann_id = 1
    for img_id, it in enumerate(items, start=1):
        with PImage.open(it["img"]) as im:
            W, H = im.size
        stem_to_id[it["stem"]] = img_id
        images.append({"id": img_id, "file_name": it["img"].name, "width": W, "height": H})
        for line in it["label"].read_text().splitlines():
            v = line.split()
            if len(v) < 7 or (len(v) - 1) % 2:
                continue
            c = int(v[0])
            coords = [float(x) for x in v[1:]]
            xs = [min(1.0, max(0.0, coords[i])) * W for i in range(0, len(coords), 2)]
            ys = [min(1.0, max(0.0, coords[i])) * H for i in range(1, len(coords), 2)]
            seg = [x for pair in zip(xs, ys) for x in pair]
            area = 0.5 * abs(sum(xs[i] * ys[(i + 1) % len(xs)] - xs[(i + 1) % len(xs)] * ys[i]
                                 for i in range(len(xs))))
            x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
            if area <= 1 or x1 <= x0 or y1 <= y0:
                continue
            annotations.append({"id": ann_id, "image_id": img_id, "category_id": c + 1,
                                "segmentation": [seg], "area": float(area),
                                "bbox": [x0, y0, x1 - x0, y1 - y0], "iscrowd": 0})
            ann_id += 1
    coco = {"info": {"description": "Session 19 YOLO11x-seg evaluation"},
            "licenses": [], "images": images, "annotations": annotations,
            "categories": [{"id": i + 1, "name": n, "supercategory": "skin"}
                           for i, n in enumerate(names)]}
    dest.write_text(json.dumps(coco))
    return coco, stem_to_id


def dets_to_coco(pred_per_stem: dict[str, list], stem_to_id: dict[str, int],
                 dest: Path) -> Path:
    from pycocotools import mask as maskutil
    results = []
    for stem, dets in pred_per_stem.items():
        img_id = stem_to_id.get(stem)
        if img_id is None:
            continue
        for d in dets:
            m = np.asfortranarray(d["mask"].astype(np.uint8))
            rle = maskutil.encode(m)
            rle["counts"] = rle["counts"].decode("ascii")
            results.append({"image_id": img_id, "category_id": d["cls"] + 1,
                            "score": float(d["score"]), "segmentation": rle})
    dest.write_text(json.dumps(results))
    return dest


def coco_segm_eval(gt_json: Path, dt_json: Path, names: list[str]) -> dict[str, Any]:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    coco_gt = COCO(str(gt_json))
    if dt_json.stat().st_size <= 2 or json.loads(dt_json.read_text()) == []:
        return {"mAP50": 0.0, "mAP50_95": 0.0, "per_class_map50": {}}
    coco_dt = coco_gt.loadRes(str(dt_json))
    ev = COCOeval(coco_gt, coco_dt, "segm")
    ev.evaluate(); ev.accumulate(); ev.summarize()
    overall_map50 = float(ev.stats[1])
    overall_map = float(ev.stats[0])
    per_class = {}
    for ci, name in enumerate(names, start=1):
        e = COCOeval(coco_gt, coco_dt, "segm")
        e.params.catIds = [ci]
        e.evaluate(); e.accumulate(); e.summarize()
        per_class[names[ci - 1]] = float(e.stats[1])
    return {"mAP50": overall_map50, "mAP50_95": overall_map,
            "per_class_map50": per_class}


# --------------------------------------------------------------------------- #
# main
# --------------------------------------------------------------------------- #
def main() -> int:
    ap = argparse.ArgumentParser(
        description="YOLO11x-seg 5-fold CV + multi-seed + ensembling. "
                    "All defaults are tuned for Kaggle T4x2 'Save and Run All' — "
                    "no arguments required."
    )
    ap.add_argument("--data-root", default="/kaggle/working/Image_Annotation_rebuilt",
                    help="Rebuilt dataset root (train/valid/test each with images+labels).")
    ap.add_argument("--work-dir", default="/kaggle/working/session19_yolo11x_cv")
    ap.add_argument("--model", default="yolo11x-seg.pt")
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--epochs", type=int, default=40,
                    help="Max epochs per fold (upper bound; fold-hours=1.8 is the primary time cap).")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=0,
                    help="Global training batch. 0 = automatic Ultralytics batch selection.")
    ap.add_argument("--predict-batch", type=int, default=4,
                    help="Conservative single-GPU inference batch for held-out predictions.")
    ap.add_argument("--device", default="auto",
                    help="'auto' detects GPUs automatically (default). "
                         "Override: '0,1' T4x2, '0' single GPU, 'cpu'.")
    ap.add_argument("--cv-seed", type=int, default=42)
    ap.add_argument("--seeds", type=int, nargs="+", default=[42],
                    help="Seeds for multi-seed stability (fold 0). >1 value enables Phase C.")
    ap.add_argument("--patience", type=int, default=10)
    ap.add_argument("--cache", default="False",
                    help="'disk', 'ram', or 'False'. Default=False to avoid Kaggle disk-quota crashes.")
    ap.add_argument("--close-mosaic", type=int, default=10)
    ap.add_argument("--copy-paste", type=float, default=0.5)
    ap.add_argument("--mosaic", type=float, default=1.0)
    ap.add_argument("--degrees", type=float, default=0.0)
    # inference / fusion
    ap.add_argument("--conf", type=float, default=0.001)
    ap.add_argument("--pred-iou", type=float, default=0.7)
    ap.add_argument("--wbf-iou", type=float, default=0.55)
    ap.add_argument("--min-support", type=int, default=1)
    ap.add_argument("--session-hours", type=float, default=11.0,
                    help="Hard wall-clock budget in hours. Folds are skipped if budget is exceeded.")
    ap.add_argument("--fold-hours", type=float, default=1.8,
                    help="Max training time per fold in hours (passed to Ultralytics time= arg). "
                         "5 folds x 1.8h = 9h + 1h eval = ~10h total, safely within 12h Kaggle limit.")
    ap.add_argument("--skip-existing", action="store_true", default=True,
                    help="Reuse a fold's best.pt if present (resume across sessions).")
    ap.add_argument("--no-skip-existing", dest="skip_existing", action="store_false")
    ap.add_argument("--skip-train", action="store_true",
                    help="Skip all training; ensemble from saved models.")
    ap.add_argument("--quick", action="store_true",
                    help="Smoke test: 3 epochs + 200 images.")
    # auto-rebuild is ON by default for 'Save and Run All'
    ap.add_argument("--auto-rebuild", dest="auto_rebuild", action="store_true", default=True,
                    help="Auto-run SESSION_00 if rebuilt dataset not found (default: on).")
    ap.add_argument("--no-auto-rebuild", dest="auto_rebuild", action="store_false",
                    help="Disable auto-rebuild; fail if rebuilt dataset not found.")
    args, _unknown = ap.parse_known_args()  # ignore Jupyter/papermill injected args

    # ---- auto-install required packages ------------------------------------ #
    try:
        import ultralytics  # noqa: F401
    except ImportError:
        import subprocess as _sp
        print("  Installing ultralytics ...")
        _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold  # noqa: F401
    except ImportError:
        import subprocess as _sp
        print("  Installing iterative-stratification ...")
        _sp.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "iterative-stratification"])
    try:
        from pycocotools.coco import COCO  # noqa: F401
    except ImportError:
        import subprocess as _sp
        print("  Installing pycocotools ...")
        _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools"])

    # ---- resolve device and batch ------------------------------------------ #
    if args.device == "auto":
        args.device = _auto_detect_device()
    n_gpus = len(str(args.device).split(",")) if args.device not in ("cpu", "") else 0
    if args.batch == 0:
        # Kaggle Jupyter kernel permanently holds ~6.42 GiB on GPU 0 (PID 45).
        # That leaves only 14.56 - 6.42 = 8.14 GiB for DDP rank0.
        # YOLO11x at batch=8/GPU needs ~8.11 GiB → only 32 MB free → OOM.
        # batch=6/GPU needs ~6.3 GiB → 1.84 GiB headroom → safe.
        args.batch = max(4, 6 * max(1, n_gpus))
        print(f"  [batch]  auto-set to {args.batch} ({n_gpus or 1} GPU(s) x 6)")

    if args.cache == "False":
        args.cache = False

    # Reduce CUDA memory fragmentation (recommended by PyTorch OOM error message)
    import os as _os
    _os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

    # ---- session-hours budget setup --------------------------------------- #
    import time as _time
    _session_start = _time.monotonic()

    def _budget_remaining_h() -> float:
        return args.session_hours - (_time.monotonic() - _session_start) / 3600.0

    def _over_budget() -> bool:
        remaining = _budget_remaining_h()
        if remaining <= 0:
            print(f"  [budget] Wall-clock limit {args.session_hours}h reached — "
                  f"skipping remaining folds to preserve eval time.", flush=True)
            return True
        return False

    # ---- locate (or build) the rebuilt dataset ----------------------------- #
    root = _resolve_data_root(
        args.data_root,
        auto_rebuild=args.auto_rebuild,
        rebuild_out=args.data_root,   # build into the requested output dir
    )
    main_yaml = root / "data.yaml"
    # (main_yaml is guaranteed to exist if _resolve_data_root returned)
    names = load_names(main_yaml)
    K = len(names)
    work = Path(args.work_dir).resolve()
    (work / "ensemble").mkdir(parents=True, exist_ok=True)

    train_items = gather_split(root, "train")
    valid_items = gather_split(root, "valid")
    test_items = gather_split(root, "test")
    trainval = train_items + valid_items
    if not trainval:
        raise SystemExit("No train/valid images found under the rebuilt dataset.")
    if args.quick:
        random.Random(0).shuffle(trainval)
        trainval = trainval[:200]
        test_items = test_items[:60]
        args.epochs = 3
    print(f"Rebuilt root : {root}")
    print(f"Classes ({K}): {names}")
    print(f"Trainval pool: {len(trainval)} images | Test holdout: {len(test_items)} images")

    # ---- build folds from trainval (test held out) ------------------------ #
    Y = multi_hot(trainval, K)
    fold_of = repair_presence(stratified_kfold(Y, args.folds, args.cv_seed), Y, args.folds)
    fold_yamls = write_fold_files(work, trainval, fold_of, args.folds, names)
    print("Fold valid sizes:",
          {f: int((fold_of == f).sum()) for f in range(args.folds)})

    summary: dict[str, Any] = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "model": args.model, "classes": names, "data_root": str(root),
        "config": {k: getattr(args, k) for k in
                   ("folds", "epochs", "imgsz", "batch", "device", "cv_seed",
                    "seeds", "conf", "wbf_iou", "min_support")},
        "phaseA_cv": [], "phaseB_ensemble": {}, "phaseC_seeds": [],
    }
    summary_path = work / "session19_summary.json"
    previous_summary: dict[str, Any] = {}
    if args.skip_existing and summary_path.is_file():
        try:
            previous_summary = json.loads(summary_path.read_text())
        except (OSError, json.JSONDecodeError) as error:
            print(f"  [resume] Existing summary could not be read: {error}")

    def save():
        summary_path.write_text(json.dumps(summary, indent=2, default=str))

    # ---- Phase A: cross-validation --------------------------------------- #
    print("\n" + "=" * 64 + "\nPHASE A - 5-fold cross-validation\n" + "=" * 64)
    fold_models: list[str] = []
    resume_models = [work / "runs" / f"fold_{f}_seed{args.cv_seed}" / "weights" / "best.pt"
                     for f in range(args.folds)]
    previous_phase_a = previous_summary.get("phaseA_cv", [])
    can_resume_phase_a = (
        args.skip_existing
        and len(previous_phase_a) == args.folds
        and all(path.is_file() for path in resume_models)
        and previous_summary.get("model") == args.model
        and previous_summary.get("classes") == names
        and previous_summary.get("config", {}).get("cv_seed") == args.cv_seed
    )
    if can_resume_phase_a:
        print("  [resume] Reusing completed Phase A summary and fold checkpoints.")
        fold_models = [str(path) for path in resume_models]
        summary["phaseA_cv"] = previous_phase_a
        for key in ("phaseA_cv_mean_map50", "phaseA_cv_std_map50"):
            if key in previous_summary:
                summary[key] = previous_summary[key]
        summary["resumed_phase_a"] = True
    elif not args.skip_train:
        for f, fy in enumerate(fold_yamls):
            if _over_budget():
                print(f"  [budget] Skipping fold {f} and all remaining folds.", flush=True)
                break
            args.seed_for_run = args.cv_seed
            print(f"\n--- Fold {f}/{args.folds - 1} (seed {args.cv_seed}) "
                  f"| budget remaining: {_budget_remaining_h():.2f}h ---")
            res = train_fold(args.model, fy, f"fold_{f}_seed{args.cv_seed}", args)
            res["fold"] = f
            summary["phaseA_cv"].append(res)
            fold_models.append(res["best_pt"])
            print(f"  fold {f}: val mask mAP50 = {res['val_map50']:.4f}")
            save()
            # Clear GPU cache so DDP workers from this fold fully release memory
            # before the next fold spawns its own DDP workers.
            try:
                import gc, torch
                torch.cuda.empty_cache()
                gc.collect()
                import time as _t; _t.sleep(3)
                print(f"  [mem] GPU cache cleared after fold {f}.", flush=True)
            except Exception:
                pass
        if summary["phaseA_cv"]:
            arr = np.array([r["val_map50"] for r in summary["phaseA_cv"]], dtype=float)
            summary["phaseA_cv_mean_map50"] = float(np.nanmean(arr))
            summary["phaseA_cv_std_map50"] = float(np.nanstd(arr))
            print(f"\nCV mask mAP50 = {np.nanmean(arr):.4f} +/- {np.nanstd(arr):.4f}")
            save()
    else:
        for f in range(args.folds):
            bp = work / "runs" / f"fold_{f}_seed{args.cv_seed}" / "weights" / "best.pt"
            if bp.is_file():
                fold_models.append(str(bp))
        print(f"--skip-train: using {len(fold_models)} saved fold models.")

    # ---- Phase B: fold ensembling on held-out test ----------------------- #
    if test_items and fold_models:
        print("\n" + "=" * 64 + "\nPHASE B - fold ensembling on held-out test\n" + "=" * 64)
        gt_json = work / "ensemble" / "coco_gt_test.json"
        _, stem_to_id = build_coco_gt(test_items, names, gt_json)
        cached_solo = [work / "ensemble" / f"coco_dt_fold{f}.json"
                       for f in range(len(fold_models))]
        cached_ensemble = work / "ensemble" / "coco_dt_ensemble.json"
        reuse_predictions = (args.skip_existing and cached_ensemble.is_file()
                             and all(path.is_file() for path in cached_solo))

        if reuse_predictions:
            print("  [resume] Reusing completed fold and ensemble COCO predictions.")
            single_scores = []
            for f, dt in enumerate(cached_solo):
                m = coco_segm_eval(gt_json, dt, names)
                single_scores.append(m["mAP50"])
                print(f"    cached fold {f} solo test mAP50 = {m['mAP50']:.4f}")
            ens = coco_segm_eval(gt_json, cached_ensemble, names)
        else:
            preds_by_model = []
            single_scores = []
            for f, bp in enumerate(fold_models):
                print(f"  predicting test with fold {f} ...")
                preds = predict_instances(bp, test_items, args)
                preds_by_model.append(preds)
                # solo model test AP (mask): fuse of a single model = its own dets
                solo = {stem: fuse_image([preds.get(stem, [])], 1, args)
                        for stem in [it["stem"] for it in test_items]}
                dt = dets_to_coco(solo, stem_to_id,
                                  work / "ensemble" / f"coco_dt_fold{f}.json")
                m = coco_segm_eval(gt_json, dt, names)
                single_scores.append(m["mAP50"])
                print(f"    fold {f} solo test mAP50 = {m['mAP50']:.4f}")

            # ensemble across all folds
            print("  fusing all folds ...")
            fused = {}
            for it in test_items:
                stem = it["stem"]
                fused[stem] = fuse_image([pm.get(stem, []) for pm in preds_by_model],
                                         len(fold_models), args)
            dt_ens = dets_to_coco(fused, stem_to_id, cached_ensemble)
            ens = coco_segm_eval(gt_json, dt_ens, names)
        summary["phaseB_ensemble"] = {
            "single_model_test_map50": single_scores,
            "single_model_mean_map50": float(np.mean(single_scores)) if single_scores else None,
            "ensemble_test_map50": ens["mAP50"],
            "ensemble_test_map50_95": ens["mAP50_95"],
            "ensemble_per_class_map50": ens["per_class_map50"],
            "ensemble_gain_over_mean": (ens["mAP50"] - float(np.mean(single_scores)))
            if single_scores else None,
            "reused_cached_predictions": reuse_predictions,
        }
        print(f"\n  single-model mean test mAP50 = {np.mean(single_scores):.4f}")
        print(f"  ENSEMBLE test mAP50          = {ens['mAP50']:.4f}")
        print(f"  per-class: {ens['per_class_map50']}")
        save()

    # ---- Phase C: multi-seed stability (fold 0) -------------------------- #
    extra_seeds = [s for s in args.seeds if s != args.cv_seed]
    if extra_seeds and not args.skip_train and test_items:
        print("\n" + "=" * 64 + "\nPHASE C - multi-seed stability (fold 0)\n" + "=" * 64)
        gt_json = work / "ensemble" / "coco_gt_test.json"
        if not gt_json.is_file():
            _, stem_to_id = build_coco_gt(test_items, names, gt_json)
        else:
            stem_to_id = {it["stem"]: i + 1 for i, it in enumerate(test_items)}
        seed_rows = []
        # include the cv-seed fold-0 test score for a complete table
        base = next((r for r in summary["phaseA_cv"] if r.get("fold") == 0), None)
        for s in [args.cv_seed] + extra_seeds:
            args.seed_for_run = s
            name = f"fold_0_seed{s}"
            res = train_fold(args.model, fold_yamls[0], name, args)
            preds = predict_instances(res["best_pt"], test_items, args)
            solo = {it["stem"]: fuse_image([preds.get(it["stem"], [])], 1, args)
                    for it in test_items}
            dt = dets_to_coco(solo, stem_to_id, work / "ensemble" / f"coco_dt_{name}.json")
            m = coco_segm_eval(gt_json, dt, names)
            seed_rows.append({"seed": s, "val_map50": res["val_map50"],
                              "test_map50": m["mAP50"]})
            print(f"  seed {s}: val mAP50 {res['val_map50']:.4f} | test mAP50 {m['mAP50']:.4f}")
            summary["phaseC_seeds"] = seed_rows
            save()
        tv = np.array([r["test_map50"] for r in seed_rows], dtype=float)
        summary["phaseC_seed_test_mean"] = float(np.nanmean(tv))
        summary["phaseC_seed_test_std"] = float(np.nanstd(tv))
        print(f"\n  seed test mAP50 = {np.nanmean(tv):.4f} +/- {np.nanstd(tv):.4f}")
        save()

    # ---- final summary ---------------------------------------------------- #
    save()
    print("\n" + "=" * 64 + "\nSESSION 19 COMPLETE\n" + "=" * 64)
    if "phaseA_cv_mean_map50" in summary:
        print(f"CV (val)  mask mAP50 : {summary['phaseA_cv_mean_map50']:.4f} "
              f"+/- {summary['phaseA_cv_std_map50']:.4f}")
    if summary["phaseB_ensemble"]:
        b = summary["phaseB_ensemble"]
        print(f"Test single-mean     : {b['single_model_mean_map50']:.4f}")
        print(f"Test ENSEMBLE mAP50  : {b['ensemble_test_map50']:.4f}  "
              f"(gain {b['ensemble_gain_over_mean']:+.4f})")
    if summary.get("phaseC_seeds"):
        print(f"Seed stability (test): {summary['phaseC_seed_test_mean']:.4f} "
              f"+/- {summary['phaseC_seed_test_std']:.4f}")
    print(f"\nSummary : {summary_path}")
    print(f"Models  : {work / 'runs'}")
    return 0


if __name__ == "__main__":
    import textwrap as _tw

    # ── Notebook / IPython DDP fix ─────────────────────────────────────────
    # When this script is pasted directly into a Kaggle notebook cell, Python
    # stores it at a temp path like /tmp/ipykernel_24/1582043190.py.
    # Ultralytics DDP (torch.distributed.run) launches a subprocess that must
    # import this file by its real path — the temp path is not importable, so
    # DDP crashes with exit-code 1 (CalledProcessError).
    #
    # Fix: if we are running inside an IPython/Jupyter kernel, write the full
    # script source to a stable path on disk and re-run it as a real subprocess.
    # The child process has a proper __file__ so DDP works correctly.
    # ──────────────────────────────────────────────────────────────────────────
    _this_file = Path(globals().get("__file__", "")).resolve()
    _is_notebook = (
        not _this_file.is_file()
        or "ipykernel" in str(_this_file)
        or "tmp" in str(_this_file).lower()
    )

    if _is_notebook:
        import subprocess as _sp
        _runtime = Path("/kaggle/working/SESSION_19_runtime.py")
        _src = ""
        _src_errors: list[str] = []

        # ── Strategy 1: Read the temp ipykernel file directly (MOST RELIABLE)
        # The file IS on disk at _this_file — DDP just can't import it by that
        # temp path. We read it and write it to a stable path ourselves.
        if _this_file.is_file():
            try:
                _raw = _this_file.read_text(encoding="utf-8", errors="replace")
                _candidate = _tw.dedent(_raw).lstrip()
                if "def main" in _candidate and "train_fold" in _candidate:
                    _src = _candidate
                    print(f"[notebook] Source read from kernel file: {_this_file}",
                          flush=True)
            except Exception as _e:
                _src_errors.append(f"kernel-file read: {_e}")

        # ── Strategy 2: IPython history (fallback when kernel file unreadable)
        if not _src:
            try:
                from IPython import get_ipython as _gip
                _shell = _gip()
                _hist = getattr(getattr(_shell, "history_manager", None),
                               "input_hist_raw", [])
                for _cell in reversed(_hist):
                    if "def main" in _cell and "train_fold" in _cell:
                        _candidate = _tw.dedent(_cell).lstrip()
                        _src = _candidate
                        print("[notebook] Source recovered from IPython history.",
                              flush=True)
                        break
            except Exception as _e:
                _src_errors.append(f"IPython history: {_e}")

        # ── Strategy 3: reuse existing runtime if build marker matches
        if not _src and _runtime.is_file():
            _candidate = _runtime.read_text(encoding="utf-8", errors="replace")
            if f'RUNTIME_BUILD = "{RUNTIME_BUILD}"' in _candidate:
                _src = _candidate
                print(f"[notebook] Reusing existing runtime (build marker matches).",
                      flush=True)

        if not _src:
            raise RuntimeError(
                "Session 19: could not recover script source to write the DDP runtime.\n"
                f"Errors: {'; '.join(_src_errors) or 'none'}\n"
                "Fix: upload SESSION_19_yolo11x_seg_cv_ensemble.py as a Kaggle dataset "
                "and run:\n  !python /kaggle/input/.../SESSION_19_yolo11x_seg_cv_ensemble.py"
            )
        try:
            compile(_src, str(_runtime), "exec")
        except SyntaxError as _e:
            raise RuntimeError(f"Recovered source has a syntax error: {_e}") from _e
        _runtime.parent.mkdir(parents=True, exist_ok=True)
        _temporary = _runtime.with_suffix(".py.tmp")
        _temporary.write_text(_src, encoding="utf-8")
        _temporary.replace(_runtime)
        print(f"[notebook] Wrote runtime to {_runtime}", flush=True)
        print(f"[notebook] Re-launching as subprocess for DDP support ...", flush=True)
        # Pass through any non-Jupyter argv (filters out Jupyter-injected args)
        _clean_argv = [a for a in sys.argv[1:]
                       if not a.startswith(("-f", "--f")) and "jupyter" not in a.lower()]
        _command = [sys.executable, "-u", str(_runtime)] + _clean_argv
        _log = Path("/kaggle/working/SESSION_19_runtime_console.log")
        print(f"[notebook] Runtime log: {_log}", flush=True)
        with _log.open("w", encoding="utf-8") as _stream:
            _process = _sp.Popen(_command, stdout=_sp.PIPE, stderr=_sp.STDOUT,
                                 text=True, bufsize=1)
            assert _process.stdout is not None
            for _line in _process.stdout:
                _stream.write(_line)
                _stream.flush()
                print(_line, end="", flush=True)
            _rc = _process.wait()
        # Raise a normal notebook exception for a child-process failure so
        # Kaggle/Papermill exposes the real run as failed without killing the
        # kernel through SystemExit.
        if _rc:
            _tail = _log.read_text(encoding="utf-8", errors="replace").splitlines()[-120:]
            print("\n" + "=" * 78 + "\nSESSION 19 CHILD LOG TAIL\n" + "=" * 78,
                  flush=True)
            print("\n".join(_tail), flush=True)
            raise RuntimeError(f"Session 19 runtime failed with exit status {_rc}; log: {_log}")
        else:
            print(f"\n[notebook] Session 19 subprocess completed successfully.",
                  flush=True)
    else:
        # Running as a normal script (via !python ...) — DDP works fine.
        # NOTE: Do NOT use os._exit() here — it kills the Jupyter kernel when
        # run via %run. sys.exit(0) raises SystemExit(0) which is harmless in a
        # subprocess (!python ...) and is caught silently when run inline.
        _rc = main()
        try:
            sys.exit(_rc)
        except SystemExit as _e:
            if _e.code:   # non-zero = real failure, re-raise
                raise


[notebook] Source recovered from IPython history.
[notebook] Wrote runtime to /kaggle/working/SESSION_19_runtime.py
[notebook] Re-launching as subprocess for DDP support ...
[notebook] Runtime log: /kaggle/working/SESSION_19_runtime_console.log
  Installing ultralytics ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 4.9 MB/s eta 0:00:00
  Installing iterative-stratification ...
  [device] 2 GPUs detected → using '0,1' (DDP)
  [batch]  auto-set to 12 (2 GPU(s) x 6)
  [auto-detect] Found rebuilt dataset at: /kaggle/input/datasets/munnamuntasirul/skin-disease-yolo/skin_disease_yolo
Rebuilt root : /kaggle/input/datasets/munnamuntasirul/skin-disease-yolo/skin_disease_yolo
Classes (19): ['acne', 'Darier_s-Disease', 'Epidermolysis-Bullosa-Pruriginosa', 'Hailey-Hailey-Disease', 'Hemangiome', 'Impetigo', 'Leishmanios', 'Lupus-E